# Quickstart of *SaivDr Package* (Python)

A brief introduction to image restoration with *SaivDr Package*.

This notebook is the Python (PyTorch) counterpart of the live script
[`quickstart.m`](quickstart.m). The sections correspond to those of the live script.

## Summary

*SaivDr* is an abbreviation of *Sparsity-Aware Image and Volume Data
Restoration*. This package is developed for

* Experiments,
* Development and
* Implementation

of sparsity-aware image and volume data restoration algorithms.

In particular, this package provides a rich set of classes related to
non-separable oversampled lapped transforms (*NSOLTs*), which allow for
convolutional layers with

* Parseval tight (paraunitary),
* Symmetric and
* Multiresolution

properties. The Python package `saivdr` provides the NSOLT layers and
networks for PyTorch, together with the degradation processes and the
ISTA-based restoration used below. See [`python/README.md`](python/README.md)
for the details.

## Install the package

Run the following once from the top directory of this repository
(see [`python/README.md`](python/README.md) for other options):

```sh
pip install "./python[examples]"
```

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import torch
from skimage import data

from saivdr.dcnn import atomic_imshow
from saivdr.dcnn.matlab_compat import load_nsoltx
from saivdr.degradation import AdditiveWhiteGaussianNoiseSystem, BlurSystem, DegradationSystem
from saivdr.restoration import IstaImRestoration2d, psnr

DATA = pathlib.Path("python/examples/data")  # NSOLTs designed in MATLAB
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_dtype(torch.float64)
print("device:", device)

## Read a source image

In the following, an image restoration procedure with this package is
described. As a preliminary, let us read an RGB picture as the source image.

`peppers.png` of MATLAB is not available in Python, so a 256×256 crop of
`skimage.data.astronaut()` is used instead.

In [ ]:
width = 256  # Width
height = 256  # Height
px = 128  # Horizontal position of cropping
py = 128  # Vertical position of cropping
srcImg = data.astronaut().astype(np.float64) / 255
orgImg = torch.from_numpy(srcImg[py : py + height, px : px + width].copy())
orgImg = orgImg.permute(2, 0, 1).unsqueeze(0).to(device)  # 1 x 3 x H x W


def to_image(t):
    # 1 x C x H x W tensor -> H x W x C array for imshow
    return t.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()

## Create a degradation system object

Suppose that we only have a degraded image $\mathbf{x}$ which is
contaminated by Gaussian kernel blur $\mathbf{P}$ with additive white
Gaussian noise $\mathbf{w}$, i.e.

$$\mathbf{x} = \mathbf{Pu} + \mathbf{w},$$

where $\mathbf{u}$ is an unknown clean original image.

Let us generate a degradation process by using the following classes
provided by *SaivDr Package*.

* `saivdr.degradation.BlurSystem`
* `saivdr.degradation.AdditiveWhiteGaussianNoiseSystem`
* `saivdr.degradation.DegradationSystem`

In [ ]:
blur = BlurSystem(  # Instantiation of blur process
    blur_type="Gaussian",  # Blur type
    sigma_of_gaussian_kernel=2,  # Sigma for Gaussian kernel
    boundary_option="Symmetric",  # Boundary option
)

nsigma = 5  # Sigma for AWGN for scale [0..255]
noise_var = (nsigma / 255) ** 2  # Normalize sigma to scale [0..1]
awgn = AdditiveWhiteGaussianNoiseSystem(  # Instantiation of AWGN
    mean=0,
    variance=noise_var,
    generator=torch.Generator(device=device).manual_seed(0),
)

dgrd = DegradationSystem(blur, awgn)  # Integration of blur and AWGN

## Generate an observed image

Then, let us generate an observed image $\mathbf{x}$ by the
`DegradationSystem` object, `dgrd`, created in the previous step.

In [ ]:
obsImg = dgrd(orgImg)

## Create an NSOLT

In order to restore the clean image $\mathbf{u}$, let us assume that
$\mathbf{u}$ is represented by a dictionary $\mathbf{D}$ as

$$\mathbf{u} = \mathbf{Dy},$$

where $\mathbf{y}$ is a coefficient vector and expected to be sparse, i.e.
the number of non-zero coefficients is small.

A non-separable oversampled lapped transform (NSOLT) can be used as a
dictionary $\mathbf{D}$. In this deblurring demo, the pre-designed NSOLT
which `quickstart.m` loads from `examples/quickdesign/results/` is used.
It was exported from MATLAB to `python/examples/data` by

```matlab
q = load('examples/quickdesign/results/nsolt_d2x2_c4+4_o4+4_v1_l4_n2048_peppers128x128.mat');
saivdr_export_nsoltx(q.nsolt, 'nsoltx_quickdesign_peppers128.mat')
```

If you are interested in the design of NSOLT, please refer, for example, to
[`python/examples/notebooks/quickdesign.ipynb`](python/examples/notebooks/quickdesign.ipynb).

In [ ]:
# Parameters for NSOLT
nLevels = 4  # of wavelet tree levels
# Decimation factor [2 2], # of channels [4 4], polyphase order [4 4] and
# # of vanishing moments 1 are given in the MAT-file.
nsoltFile = DATA / "nsoltx_quickdesign_peppers128.mat"

# Show the atomic images, as nsolt.atmimshow() in MATLAB
fig, ax = plt.subplots(figsize=(5, 3))
atomic_imshow(load_nsoltx(nsoltFile, mode="Synthesizer", number_of_levels=1), patch_size=(16, 16), ax=ax)
ax.set_title("Atomic images of NSOLT");

## Create an analysis and synthesis system

`load_nsoltx` builds the NSOLT network, which consists of the analyzer
$\mathbf{D}^T$ and the synthesizer $\mathbf{D}$. Their coefficients agree
with those of `Analysis2dSystem` and `Synthesis2dSystem` in MATLAB.

In [ ]:
nsolt = load_nsoltx(nsoltFile, number_of_levels=nLevels, device=device)
analyzer, synthesizer = nsolt.analyzer, nsolt.synthesizer

## Create an ISTA-based image restoration system object

Let the $\ell_1$-norm of $\mathbf{y}$, i.e. $\|\mathbf{y}\|_1$, be the
sparsity measure of the coefficients. Then, the deblurring problem can be
formulated as

$$\hat{\mathbf{y}} = \arg\min_{\mathbf{y}}\frac{1}{2}\|\mathbf{x}-\mathbf{PDy}\|_2^2+\lambda\|\mathbf{y}\|_1.$$

Provided that the blur process $\mathbf{P}$ is known and given as a linear
process, the iterative shrinkage/thresholding algorithm (ISTA) becomes
applicable to solve the above problem. In *SaivDr Package*, the following
class is available:

* `saivdr.restoration.IstaImRestoration2d`

In [ ]:
lambda_ = 0.00185  # lambda
ista = IstaImRestoration2d(
    synthesizer=synthesizer,  # Synthesizer (Dictionary)
    adj_of_synthesizer=analyzer,  # Analyzer (Adj. of dictionary)
    linear_process=blur,  # Blur process
    lambda_=lambda_,  # Parameter lambda
    eps0=1e-6,
)

## Create a step monitor

ISTA iteratively approaches the optimum solution. In order to observe the
intermediate results, a function can be given as `step_monitor`, which
plays the role of `StepMonitoringSystem` in MATLAB. It is called as
`step_monitor(n_iter, restored)` after every iteration and records the PSNR
here.

In [ ]:
psnrs = []


def stepmonitor(nItr, resImg):
    psnrs.append(psnr(resImg.clamp(0, 1), orgImg))


# Set the function to the ISTA object
ista.step_monitor = stepmonitor

## Perform ISTA-based image restoration

Calling the ISTA object, `ista`, executes the ISTA-based image restoration
to deblur the observed image. As the result, a restored image

$$\hat{\mathbf{u}} = \mathbf{D}\hat{\mathbf{y}}$$

is obtained.

In [ ]:
with torch.no_grad():
    resImg = ista(obsImg)

## Extract the final evaluation

The step monitor stores the PSNR calculated at every iteration of ISTA. The
number of iterations is given by `ista.n_iter`.

In [ ]:
nItr = ista.n_iter
psnr_ista = psnrs[-1]
print(f"ISTA: {nItr} iterations, PSNR {psnr_ista:.2f} dB")

plt.plot(range(1, nItr + 1), psnrs)
plt.xlabel("Iteration")
plt.ylabel("PSNR [dB]")
plt.title("ISTA");

## Perform Wiener filtering

As a reference, let us show a result of Wiener filter. The following
function performs the Wiener deconvolution in the frequency domain, as
`deconvwnr` of Image Processing Toolbox in MATLAB.

In [ ]:
def deconvwnr(img, psf, nsr):
    # Optical transfer function of psf for the image size (as psf2otf)
    h, w = img.shape[-2:]
    otf = torch.zeros(h, w, dtype=img.dtype, device=img.device)
    kh, kw = psf.shape
    otf[:kh, :kw] = psf
    otf = torch.roll(otf, shifts=(-(kh // 2), -(kw // 2)), dims=(0, 1))
    H = torch.fft.fft2(otf)
    G = H.conj() / (H.abs() ** 2 + nsr)
    return torch.fft.ifft2(torch.fft.fft2(img) * G).real


# Use the same blur kernel as that applied to the observed image, obsImg
blurKernel = blur.blur_kernel.to(device)

# Estimation of noise to signal ratio
nsr = noise_var / orgImg.var().item()

wnfImg = deconvwnr(obsImg, blurKernel, nsr)

# Evaluation
psnr_wfdc = psnr(wnfImg.clamp(0, 1), orgImg)
print(f"Wiener: PSNR {psnr_wfdc:.2f} dB")

## Compare deblurring performances

In order to compare the deblurring performances between two methods,
ISTA-based deblurring with NSOLT and Wiener filter, let us show the
original, observed and two results in one figure together.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
images = [orgImg, obsImg, resImg, wnfImg]
titles = [
    r"Original image $\mathbf{u}$",
    r"Observed image $\mathbf{x}$",
    rf"$\hat{{\mathbf{{u}}}}$ by ISTA: {psnr_ista:.2f} [dB]",
    rf"$\hat{{\mathbf{{u}}}}$ by Wiener: {psnr_wfdc:.2f} [dB]",
]
for ax, img, title in zip(axes.flat, images, titles):
    ax.imshow(to_image(img))
    ax.set_title(title)
    ax.axis("off")

## Release notes

[`RELEASENOTES.txt`](RELEASENOTES.txt) contains release notes on
*SaivDr Package*.